# Export web bundle for `viz/discterry` (Discterry)

Writes `nodes.parquet`, `edges.parquet`, and `meta.json` into `viz/discterry/public/data/`.

Run with cwd `notebooks/dmercator/` so `dmercator_io` imports. Set `DMERCATOR_RUN` or edit `RUN_SUBDIR` like other notebooks.


In [ ]:
import json
import os

import pyarrow as pa
import pyarrow.parquet as pq

import dmercator_io as dm

RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", "d2")
paths = dm.paths_for_run(RUN_SUBDIR)
_, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
x, y = dm.ortho_xy_disk(df)

repo = dm.repo_root()
out = repo / "viz" / "discterry" / "public" / "data"
out.mkdir(parents=True, exist_ok=True)

vertices = [str(v).strip() for v in df["Vertex"].tolist()]
name_to_i = {name: i for i, name in enumerate(vertices)}

nodes_tbl = pa.table({
    "vertex": pa.array(vertices, type=pa.string()),
    "x": pa.array(x.astype("float32"), type=pa.float32()),
    "y": pa.array(y.astype("float32"), type=pa.float32()),
})

src, dst = [], []
for u, v in G.edges():
    uu, vv = str(u).strip(), str(v).strip()
    if uu not in name_to_i or vv not in name_to_i:
        continue
    iu, iv = name_to_i[uu], name_to_i[vv]
    src.append(iu)
    dst.append(iv)

edges_tbl = pa.table({
    "src": pa.array(src, type=pa.int32()),
    "dst": pa.array(dst, type=pa.int32()),
})

pq.write_table(nodes_tbl, out / "nodes.parquet")
pq.write_table(edges_tbl, out / "edges.parquet")

meta = {
    "run_subdir": RUN_SUBDIR,
    "n_vertices": len(vertices),
    "n_edges": len(src),
    "default_focus": "STAC3" if "STAC3" in name_to_i else (vertices[0] if vertices else ""),
    "default_seeds": [
        "CATSPER1", "STAC3", "CACNB3", "CACNA1C", "CACNA1S",
    ],
}
with open(out / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Wrote nodes:", out / "nodes.parquet", "n_vertices=", len(vertices))
print("Wrote edges:", out / "edges.parquet", "n_edges=", len(src))
print("Wrote meta:", out / "meta.json")
